# 02 - Preprocesado LPMC

Replica el script del profe `0-Process-LPMC.py`. Hace dummies de `purpose`, agrega fueltype mas general y mapea `travel_mode`, borra columnas no usadas y divide train/test por `survey_year`.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
RAW_PATH = ROOT / 'data' / 'raw' / 'LPMC_dataset.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'LPMC_processed.csv'
PRE_DIR = ROOT / 'data' / 'preprocessed'
PRE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)


In [2]:
df_raw = pd.read_csv(RAW_PATH)
print('Forma inicial:', df_raw.shape)


Forma inicial: (81086, 36)


In [3]:
df = df_raw.copy()

purpose_df = pd.get_dummies(df['purpose'], prefix='purpose')
df = df.join(purpose_df).drop(columns=['purpose'])

df = df.drop(columns=['faretype'])

fuel_map = {
    'Petrol_Car': 'Petrol',
    'Petrol_LGV': 'Petrol',
    'Diesel_Car': 'Diesel',
    'Diesel_LGV': 'Diesel',
    'Hybrid_Car': 'Hybrid',
    'Average_Car': 'Average',
}
df['fueltype'] = df['fueltype'].map(fuel_map)
fueltype_df = pd.get_dummies(df['fueltype'], prefix='fueltype')
df = df.join(fueltype_df).drop(columns=['fueltype'])

mode_map = {'walk': 0, 'cycle': 1, 'pt': 2, 'drive': 3}
df['travel_mode'] = df['travel_mode'].map(mode_map)

cols_to_drop = [
    'trip_id', 'person_n', 'trip_n', 'travel_year', 'travel_month', 'travel_date',
    'bus_scale', 'dur_pt_total', 'dur_pt_int_total', 'cost_driving_fuel',
    'cost_driving_con_charge', 'driving_traffic_percent',
]
df = df.drop(columns=cols_to_drop)

print('Forma tras limpiar:', df.shape)
PROCESSED_PATH


Forma tras limpiar: (81086, 30)


WindowsPath('f:/TFM/lpmc/data/processed/LPMC_processed.csv')

In [4]:
df.to_csv(PROCESSED_PATH, index=False)

train_df = df[df['survey_year'].isin([1, 2])].copy()
test_df = df[df['survey_year'] == 3].copy()

train_df = train_df.drop(columns=['survey_year'])
test_df = test_df.drop(columns=['survey_year'])

train_path = PRE_DIR / 'LPMC_train.csv'
test_path = PRE_DIR / 'LPMC_test.csv'
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print('Guardados:', train_path, test_path)
print('Tamanos -> train:', train_df.shape, 'test:', test_df.shape)


Guardados: f:\TFM\lpmc\data\preprocessed\LPMC_train.csv f:\TFM\lpmc\data\preprocessed\LPMC_test.csv
Tamanos -> train: (54766, 29) test: (26320, 29)
